___
# Лабораторная работа №4
___

## Цель работы
___

Изучить архитектуру трансформеров и принципы работы механизма self-attention, а также сравнить самостоятельно реализованную модель с предобученными трансформерами семейства BERT.

___
## 1. Подготовка данных
___

Изначально требовалось импортировать библиотеки для выполнения работы.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import BertTokenizer, BertForSequenceClassification
from tqdm import tqdm
import math
from collections import Counter
import re

# Установка устройства (GPU если доступно, иначе CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используемое устройство: {device}")

# Глобальные гиперпараметры
MAX_SEQ_LEN = 128
BATCH_SIZE = 32
VOCAB_SIZE = 20000

C:\Users\CHELOVEK\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Используемое устройство: cpu


MAX_SEQ_LEN = 128: Максимальная длина текстовой последовательности (в токенах/словах). Если исходная новость короче этого значения, она дополняется специальными padding-токенами (обычно нулями), чтобы все векторы в батче были одинакового размера. Если текст длиннее, он усекается до 128 токенов. Значение 128 является оптимальным балансом между сохранением контекста новости и экономией видеопамяти при вычислении ресурсоемких матриц внимания.

BATCH_SIZE = 32: Размер мини-батча, то есть количество текстовых примеров, которые параллельно обрабатываются нейронной сетью за один проход до обновления весов алгоритмом оптимизации. Значение 32 является стандартом де-факто для многих задач NLP: оно обеспечивает стабильную сходимость градиента, не вызывая переполнения памяти на большинстве современных видеокарт.

VOCAB_SIZE = 20000: Максимальный размер словаря, используемого для обучения собственного Трансформера с нуля. Ограничение словаря 20 000 самых часто встречающихся слов обучающей выборки позволяет контролировать размер слоя Embedding и отфильтровывает из текста редкие слова, опечатки или специфичные термины, заменяя их на специальный токен неизвестного слова <UNK>.

Описание импортированных библиотек:

torch: Основная библиотека фреймворка PyTorch. Используется для создания и вычисления тензоров, а также для переноса вычислений на видеокарту (GPU).

torch.nn (nn): Модуль, содержащий базовые строительные блоки нейросетей. Из него берутся слои (Embedding, Linear, MultiheadAttention, LayerNorm) и функция потерь (CrossEntropyLoss).

torch.optim (optim): Модуль с алгоритмами оптимизации. Применяется для обновления весов нейронной сети в процессе обучения (используются алгоритмы Adam и AdamW).

Dataset, DataLoader: Классы из модуля torch.utils.data. Предназначены для организации пайплайна данных: Dataset структурирует выборку, а DataLoader автоматически разбивает её на батчи и перемешивает.

pandas (pd): Библиотека для обработки и анализа данных. Используется для удобной и быстрой загрузки датасета из CSV-файлов, а также объединения колонок с текстом.

numpy (np): Фундаментальная библиотека для научных вычислений. Применяется для работы с числовыми массивами при извлечении предсказаний (перевод тензоров обратно в массив для подсчета метрик).

accuracy_score, f1_score: Функции из модуля sklearn.metrics. Применяются для объективной численной оценки качества прогнозирования классификатора (точность и F1-мера) на тестовой выборке.

BertTokenizer, BertForSequenceClassification: Классы из библиотеки transformers (HuggingFace). Позволяют загрузить готовую архитектуру BERT с классификационной "головой" и соответствующий ей токенизатор (алгоритм разбиения текста на токены).

tqdm: Библиотека для создания визуальных индикаторов прогресса. Используется для отображения шкалы выполнения во время циклов обучения и валидации.

math: Стандартная математическая библиотека Python. Применяется для вычисления синусов, косинусов и логарифмов в модуле позиционного кодирования (PositionalEncoding).

Counter: Класс из модуля collections. Используется для быстрого подсчета частоты встречаемости слов при формировании собственного словаря (SimpleVocab).

re: Модуль для работы с регулярными выражениями. Применяется для очистки текста от лишних символов и выделения отдельных слов (простая токенизация).

Набор данных представляет собой коллекцию новостных статей, изначально собранных академической поисковой системой ComeToMyHead. В данной работе используется версия датасета (от Xiang Zhang), в которой отобраны 4 крупнейших класса. Каждый класс сбалансирован и содержит ровно 30 000 обучающих и 1 900 тестовых образцов. Общий объем выборки составляет 120 000 примеров для обучения (train.csv) и 7 600 для тестирования (test.csv). Исходные данные представлены в формате CSV и содержат индекс класса, заголовок и описание новости.

Загрузка данных: Используется библиотека pandas для чтения файлов train.csv и test.csv. Текстовые данные (заголовок и описание) объединяются в одну строку для максимального контекста. Классы в исходном датасете нумеруются от 1 до 4, код сдвигает их к диапазону от 0 до 3 для корректной работы функции потерь CrossEntropyLoss в PyTorch.
1. World (Мир) — новости мировой политики и международных событий.

2. Sports (Спорт) — спортивные новости.

3. Business (Бизнес) — новости экономики, финансов и корпораций.

4. Sci/Tech (Наука/Технологии) — новости из мира технологий, науки и IT.


In [2]:
def load_data(train_path, test_path, sample_size=None):
    train_df = pd.read_csv(train_path, header=None, names=['class', 'title', 'description'])
    test_df = pd.read_csv(test_path, header=None, names=['class', 'title', 'description'])
    
    # Пропуск первой строки, если это заголовок
    if train_df.iloc[0]['class'] == 'Class Index':
        train_df = train_df.iloc[1:]
        test_df = test_df.iloc[1:]
    
    # Объединение заголовока и описания
    train_df['text'] = train_df['title'].astype(str) + " " + train_df['description'].astype(str)
    test_df['text'] = test_df['title'].astype(str) + " " + test_df['description'].astype(str)
    
    # Приведение классов к диапазону 0-3 (в исходнике 1-4)
    train_df['class'] = train_df['class'].astype(int) - 1
    test_df['class'] = test_df['class'].astype(int) - 1
    
        
    return train_df['text'].tolist(), train_df['class'].tolist(), test_df['text'].tolist(), test_df['class'].tolist()

class SimpleVocab:
    def __init__(self, max_size=VOCAB_SIZE):
        self.pad_idx = 0
        self.unk_idx = 1
        self.word2idx = {'<PAD>': self.pad_idx, '<UNK>': self.unk_idx}
        self.idx2word = {self.pad_idx: '<PAD>', self.unk_idx: '<UNK>'}
        self.max_size = max_size
        
    def build_vocab(self, texts):
        counter = Counter()
        for text in texts:
            words = re.findall(r'\w+', text.lower())
            counter.update(words)
        
        most_common = counter.most_common(self.max_size - 2)
        for idx, (word, _) in enumerate(most_common, start=2):
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            
    def encode(self, text, max_len=MAX_SEQ_LEN):
        words = re.findall(r'\w+', text.lower())
        indices = [self.word2idx.get(w, self.unk_idx) for w in words]
        if len(indices) < max_len:
            indices += [self.pad_idx] * (max_len - len(indices))
        else:
            indices = indices[:max_len]
        return indices

Класс SimpleVocab:
Это реализация простого токенизатора на уровне слов (word-level). Трансформеры, обученные с нуля, не могут принимать текст напрямую, им нужны числа (индексы).
build_vocab: Проходит по всему обучающему тексту, разбивает его на слова (используя регулярное выражение re.findall), подсчитывает частоту каждого слова и оставляет VOCAB_SIZE самых популярных. Каждому такому слову присваивается уникальный числовой индекс.
Специальные токены: Нулевой индекс (<PAD>) резервируется для пустышек, которыми дополняются короткие тексты до одинаковой длины MAX_SEQ_LEN. Индекс 1 (<UNK>) резервируется для "неизвестных" слов — тех, которые не попали в топ популярных или встретились только в тестовой выборке.
encode: Функция, которая берет строку текста, разбивает на слова и заменяет каждое слово на его индекс из словаря. Если слово не найдено, ставит индекс <UNK>. Затем она обрезает последовательность или дополняет её <PAD> токенами до строгой длины max_len.
Этот класс выполняет фундаментальную задачу векторизации текста, подготавливая числовые массивы одинаковой размерности, которые может обрабатывать PyTorch.

___
## 2. Подготовка датасетов
___

Чтобы нейронная сеть могла обучаться на текстах, данные нужно подавать не списком Python, а специальными тензорами, разбитыми на батчи. В PyTorch за это отвечает связка классов Dataset (указывает, как достать один элемент) и DataLoader (собирает элементы в батч).

Поскольку у CustomTransformer и у BERT разные требования к формату входных данных (CustomTransformer принимает просто массив чисел, а BERT требует еще маску внимания), в коде реализовано два разных класса, наследуемых от базового torch.utils.data.Dataset.

In [3]:
class CustomDataset(Dataset):
    def __init__(self, texts, labels, vocab=None):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded = self.vocab.encode(text)
        return torch.tensor(encoded, dtype=torch.long), torch.tensor(label, dtype=torch.long)

class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_SEQ_LEN):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long)
        }

CustomDataset (Для CustomTransformer):

Класс принимает список текстов, список меток классов и ранее созданный объект словаря vocab.
В функции __getitem__ (которая вызывается для получения конкретной новости по индексу idx) происходит вызов self.vocab.encode(text). Это превращает строку текста в список чисел (индексов слов) фиксированной длины (с отсечением длинных или добавлением <PAD>).

Метод возвращает кортеж из двух PyTorch тензоров формата torch.long (64-битные целые числа): тензор текста и тензор правильного класса.

BERTDataset (Для модели BERT):

В отличие от CustomTransformer, этот класс принимает объект токенизатора BertTokenizer из библиотеки HuggingFace.

В функции __getitem__ вызывается мощный метод encode_plus. Он делает сразу несколько вещей, специфичных для BERT: разбивает слова на подслова (WordPiece), добавляет токены начала [CLS] и конца [SEP], выравнивает длину (padding, truncation) и создает attention_mask.

attention_mask (Маска внимания): Это критически важный тензор, состоящий из единиц (для реальных слов) и нулей (для паддинг-токенов). Он сообщает механизму внимания BERT, на какие токены нужно обращать внимание, а какие игнорировать (не тратить вычислительные ресурсы на <PAD> нули).

Метод возвращает Python-словарь, содержащий три тензора: сами индексы (input_ids), маску внимания (attention_mask) и метку класса (targets).

___
## 3. Архитектура собственной модели
___

Была реализована с нуля архитектура трансформера, состоящая из трех ключевых взаимосвязанных компонентов. В отличие от рекуррентных сетей (RNN), которые читают текст слово за словом, трансформер обрабатывает все слова предложения одновременно, что требует особых механизмов для понимания контекста и порядка слов. Ниже подробно описано, как работают эти механизмы в представленном коде:

PositionalEncoding (Позиционное кодирование): Поскольку механизм внимания параллелен и не имеет понятия о порядке, к каждому вектору слова (эмбеддингу) необходимо добавить информацию о его позиции в тексте. Класс вычисляет уникальную матрицу pe на основе синусов и косинусов разных частот. В методе forward эта матрица просто прибавляется к входным эмбеддингам: x + self.pe.

TransformerEncoderLayer (Единичный слой энкодера): Это ядро вычислительной логики модели. Слой включает:
Multi-Head Self-Attention: Слой nn.MultiheadAttention позволяет каждому слову посмотреть на все остальные слова в предложении и понять свою роль (например, относится слово "банк" к реке или финансам). Использование нескольких голов (heads) позволяет параллельно фокусироваться на разных аспектах (синтаксисе, смысле, окончаниях). Сюда передаются маски (attn_mask или key_padding_mask), чтобы сеть не обращала внимание на пустые токены <PAD>.
Residual Connections & LayerNorm: После механизма внимания вычисляется остаточная связь (src + dropout(src2)). То есть к исходному входу прибавляется результат внимания. Затем применяется слой нормализации (nn.LayerNorm), который стабилизирует веса и защищает сеть от затухания градиентов.
Feed-Forward Network (FFN): Вторая часть слоя — это двухслойная полносвязная нейросеть (линейный слой -> ReLU -> линейный слой). Она применяется к каждому токену независимо от других, позволяя извлечь еще более сложные нелинейные признаки. Завершается этап еще одной остаточной связью и нормализацией.
CustomTransformer (Итоговая сеть классификации): Этот класс объединяет описанные выше компоненты в полноценный пайплайн от входных индексов до предсказания классов:
Сначала слой nn.Embedding превращает каждый индекс слова в плотный числовой вектор размерности d_model (по умолчанию 128).
Затем применяется pos_encoder для добавления информации о позиции.
Данные поочередно проходят через список из num_layers (по умолчанию 2) слоев TransformerEncoderLayer. Также здесь генерируется булева маска mask = (x == 0), которая передается в слои энкодера, чтобы сеть полностью игнорировала паддинги при вычислении внимания.
Global Average Pooling: После того как энкодер обработал всю последовательность, получается 128-мерный вектор для каждого слова. Чтобы сделать одно предсказание для всего текста, модель усредняет векторы всех слов. При этом код (~mask).unsqueeze(-1).float() гарантирует, что усредняются векторы только реальных слов, а <PAD> токены исключаются из подсчета суммы и делителя (valid_lengths).
В конце применяется слой Dropout (для защиты от переобучения) и финальный линейный классификатор self.fc, который сжимает итоговый вектор текста до 4 логитов — по одному на каждый новостной класс.

In [4]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        src2, _ = self.self_attn(src, src, src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        src2 = self.linear2(self.dropout(torch.relu(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

class CustomTransformer(nn.Module):
    def __init__(self, vocab_size, num_classes, d_model=128, num_heads=4, num_layers=2, d_ff=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_encoder = PositionalEncoding(d_model)
        
        self.layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.fc = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        mask = (x == 0)
        
        out = self.embedding(x)
        out = self.pos_encoder(out)
        
        for layer in self.layers:
            out = layer(out, src_key_padding_mask=mask)
            
        mask_expanded = (~mask).unsqueeze(-1).float()
        sum_embeddings = torch.sum(out * mask_expanded, dim=1)
        valid_lengths = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        pooled = sum_embeddings / valid_lengths
        
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)
        return logits

___
## 4. Функции обучения и оценки модели
___

В этом блоке реализованы две ключевые функции: train_epoch (прогоняет один полный цикл обучения по всем данным) и evaluate (оценивает качество модели на тестовых данных, не изменяя веса). Обе функции сделаны универсальными: с помощью флага is_bert они могут работать как с нашей собственной моделью, так и с архитектурой BERT.

В начале каждой функции вызываются методы. model.train() включает режим обучения — слои Dropout начинают случайным образом отключать нейроны (чтобы избежать переобучения), а LayerNorm собирает статистику батча. model.eval() переводит сеть в режим предсказания: Dropout отключается (сеть использует все свои ресурсы), а нормализация использует накопленные ранее средние значения. В PyTorch градиенты (направления изменения весов) по умолчанию накапливаются при каждом проходе. Поэтому в начале обработки нового батча мы обязаны очистить старые градиенты, иначе модель начнет обучаться некорректно, суммируя текущие ошибки с прошлыми.

Forward Pass (Прямой проход) и отличие BERT:

Если is_bert=True, данные извлекаются из словаря (input_ids, attention_mask, targets). Удобство библиотеки HuggingFace в том, что если передать labels прямо внутрь model(...), она сама внутри себя рассчитает функцию потерь и вернет специальный объект, содержащий как loss (ошибку), так и logits (сырые предсказания). Если это наша модель, мы передаем тензор на вход model(inputs), получаем logits и вычисляем ошибку вручную через заранее заданный criterion(logits, targets) (в нашем случае это CrossEntropyLoss). loss.backward() и optimizer.step(): Ключевая часть магии нейросетей. Метод backward запускает алгоритм обратного распространения ошибки (Backpropagation) — он вычисляет производные для каждого веса в сети, показывая, насколько этот вес виноват в ошибке. Затем step() сдвигает все веса модели в нужную сторону (согласно градиенту и learning rate), чтобы на следующем батче ошибка была меньше. В функции evaluate этих строк нет, так как при оценке мы не хотим менять веса. torch.no_grad(): Контекстный менеджер, который отключает слежение за градиентами внутри функции evaluate. Это экономит огромное количество оперативной и видеопамяти при валидации, так как нам не нужно хранить граф вычислений. Расчет метрик: Из logits (4 числа-вероятности для каждого класса) выбирается индекс максимального значения с помощью torch.argmax (это и есть предсказанный класс). Тензор переносится из видеопамяти обратно в ОЗУ (.cpu().numpy()), где классические функции из sklearn подсчитывают итоговые accuracy_score (долю правильных ответов) и f1_score.

In [5]:
def train_epoch(model, dataloader, optimizer, criterion, is_bert=False):
    model.train()
    total_loss = 0
    predictions, true_labels = [], []
    
    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()
        
        if is_bert:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            targets = batch['targets'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=targets)
            loss = outputs.loss
            logits = outputs.logits
        else:
            inputs, targets = batch
            inputs, targets = inputs.to(device), targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)
            
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        predictions.extend(preds)
        true_labels.extend(targets.cpu().numpy())
        
    acc = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='macro')
    return total_loss / len(dataloader), acc, f1

def evaluate(model, dataloader, criterion, is_bert=False):
    model.eval()
    total_loss = 0
    predictions, true_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            if is_bert:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                targets = batch['targets'].to(device)
                outputs = model(input_ids, attention_mask=attention_mask, labels=targets)
                loss = outputs.loss
                logits = outputs.logits
            else:
                inputs, targets = batch
                inputs, targets = inputs.to(device), targets.to(device)
                logits = model(inputs)
                loss = criterion(logits, targets)
                
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            true_labels.extend(targets.cpu().numpy())
            
    acc = accuracy_score(true_labels, predictions)
    f1 = f1_score(true_labels, predictions, average='macro')
    return total_loss / len(dataloader), acc, f1

___
## 6. Запуск обучения и сбор значений
___

Запуск собственного трансформера:
Создается пустой словарь SimpleVocab и заполняется частотными словами из обучающей выборки (build_vocab). Данные оборачиваются в DataLoader с размером батча 32. Важно, что train_loader перемешивается (shuffle=True), чтобы модель не заучивала последовательность данных, а test_loader — нет.  Экземпляр CustomTransformer переносится на видеокарту (.to(device)). Инициализируется оптимизатор Adam со стандартным шагом обучения (Learning Rate) lr=1e-3. Модели, обучаемые с нуля, требуют достаточно высокого lr, чтобы выбраться из случайной начальной инициализации весов. Запускается цикл на 5 эпох. В каждой эпохе модель тренируется, затем оценивается, а результаты выводятся в консоль.

Запуск Fine-tuning BERT:
Загружается официальный токенизатор bert-base-uncased (версия, приводящая все символы к нижнему регистру). Создаются даталоадеры для BERT. Размер батча здесь уменьшен до 16. BERT — огромная модель (110 млн параметров), и батч 32 часто приводит к ошибке переполнения памяти видеокарты (CUDA Out Of Memory). Загружается сама архитектура BERT (BertForSequenceClassification). Готовая модель заменяет свою стандартную голову на новую линейную проекцию на 4 класса (num_labels=4). При этом веса головы инициализируются случайно, а сам ствол (энкодер) содержит предобученные веса.
Инициализируется оптимизатор AdamW с микроскопическим шагом обучения lr=2e-5. Это критически важный момент для fine-tuning (дообучения). Если использовать большой lr, как в нашей модели, мощные градиенты полностью разрушат полезные веса BERT (это называется катастрофическое забывание). Низкий lr позволяет лишь слегка подправить модель под нашу конкретную задачу.
BERT обучается всего 2 эпохи. Благодаря мощному предобучению, BERT сходится (достигает высокой точности) крайне быстро, и дальнейшее обучение может привести лишь к переобучению (overfitting) на тренировочных данных.

In [7]:
if __name__ == "__main__":
    train_texts, train_labels, test_texts, test_labels = load_data('train.csv', 'test.csv', None)
    print(f"Загружено {len(train_texts)} тренировочных и {len(test_texts)} тестовых примеров.")
    criterion = nn.CrossEntropyLoss()

    print("\nИнициализация и обучение собственной модели Трансформера")
    vocab = SimpleVocab()
    vocab.build_vocab(train_texts)
    
    train_loader_custom = DataLoader(CustomDataset(train_texts, train_labels, vocab), batch_size=BATCH_SIZE, shuffle=True)
    test_loader_custom = DataLoader(CustomDataset(test_texts, test_labels, vocab), batch_size=BATCH_SIZE)
    
    custom_model = CustomTransformer(vocab_size=VOCAB_SIZE, num_classes=4).to(device)
    optimizer_custom = optim.Adam(custom_model.parameters(), lr=1e-3)
    
    EPOCHS_CUSTOM = 5
    for epoch in range(EPOCHS_CUSTOM):
        train_loss, train_acc, train_f1 = train_epoch(custom_model, train_loader_custom, optimizer_custom, criterion)
        val_loss, val_acc, val_f1 = evaluate(custom_model, test_loader_custom, criterion)
        print(f"Epoch {epoch+1}/{EPOCHS_CUSTOM} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")

    print("\nЗагрузка и Fine-tuning предобученной модели BERT")
    bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    
    train_loader_bert = DataLoader(BERTDataset(train_texts, train_labels, bert_tokenizer), batch_size=16, shuffle=True)
    test_loader_bert = DataLoader(BERTDataset(test_texts, test_labels, bert_tokenizer), batch_size=16)
    
    bert_model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=4).to(device)
    optimizer_bert = optim.AdamW(bert_model.parameters(), lr=2e-5)
    
    EPOCHS_BERT = 2 # BERT требует меньше эпох для адаптации
    for epoch in range(EPOCHS_BERT):
        train_loss, train_acc, train_f1 = train_epoch(bert_model, train_loader_bert, optimizer_bert, criterion, is_bert=True)
        val_loss, val_acc, val_f1 = evaluate(bert_model, test_loader_bert, criterion, is_bert=True)
        print(f"Epoch {epoch+1}/{EPOCHS_BERT} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} F1: {val_f1:.4f}")

    print("\nОбучение завершено.")

Загружено 120000 тренировочных и 7600 тестовых примеров.

Инициализация и обучение собственной модели Трансформера


Training:   1%|▉                                                                     | 50/3750 [00:30<37:25,  1.65it/s]


KeyboardInterrupt: 